# 02 — Cleaning check (before vs after)
Inspects the output of `code/business_entity_resolution/src/clean_data.py` (`work/clean/*.parquet`).
Run `mine_dictionaries.py` and `clean_data.py` first. Kernel: **base (D:\miniconda)**.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from rapidfuzz.distance import Indel

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "code" / "business_entity_resolution" / "src"
sys.path.insert(0, str(SRC))
import config
from io_utils import load_gt_pairs
from normalize import Normalizer
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 250)
SHOW = ["business_name", "name_clean", "name_core", "name_alias", "name_legal",
        "business_address", "addr_clean", "addr_nums", "addr_postcode", "addr_landmark"]
print(config.CLEAN_DIR, sorted(p.name for p in config.CLEAN_DIR.glob("*.parquet")))

## 1. Dictionary sizes & examples

In [ ]:
rep = json.loads((config.DICT_DIR / "mining_report.json").read_text("utf-8"))
display(pd.DataFrame(rep["sizes"]))
nm = json.loads((config.DICT_DIR / "name_token_map.json").read_text("utf-8"))
am = json.loads((config.DICT_DIR / "addr_component_map.json").read_text("utf-8"))
native = [(k, v) for k, v in nm["india"].items() if not k.isascii()]
print(len(native), "native-script name tokens mapped, e.g.:", native[:25])
print("state/component examples:", [(k, v) for k, v in am["india"].items() if not k.isascii()][:20])

## 2. Before / after samples

In [ ]:
t2 = pd.read_parquet(config.CLEAN_DIR / "train_s2.parquet")
def show(mask, n=8, seed=0):
    display(t2[mask].sample(n, random_state=seed)[SHOW])
print("native-script names"); show(t2.f_name_native)
print("junk prefixes");       show(t2.f_name_junk)
print("aliases (DBA / formerly)"); show(t2.f_name_alias)
print("domains");             show(t2.f_name_domain)
print("landmarks");           show(t2.addr_landmark != "")

In [ ]:
# Remaining non-ASCII after cleaning should be ~0 (everything mapped or romanised)
for c in ["name_clean", "addr_clean"]:
    print(c, "non-ascii rows:", (~t2[c].str.isascii()).mean().round(5))
# How often did the anyascii fallback fire for native names (tokens not in mined map)?
N = Normalizer(config.DICT_DIR)
smp = t2[t2.f_name_native].sample(20000, random_state=1)
from normalize import base_name_tokens
toks = [t for n in smp.business_name for t in base_name_tokens(n)[0] if not t.isascii()]
cov = np.mean([t in N.name_tok["india"] for t in toks])
print(f"native name tokens covered by mined map: {cov:.2%} (rest -> anyascii fallback)")

## 3. Test set: France + native-token coverage on unseen data

In [ ]:
te2 = pd.read_parquet(config.CLEAN_DIR / "test_s2.parquet")
display(te2[te2.country == "France"].sample(10, random_state=0)[SHOW])
smp = te2[te2.f_name_native].sample(20000, random_state=1)
toks = [t for n in smp.business_name for t in base_name_tokens(n)[0] if not t.isascii()]
print(f"TEST native name tokens covered by mined map: {np.mean([t in N.name_tok['india'] for t in toks]):.2%}")

## 4. Similarity of validation-fold true pairs: raw vs cleaned

In [ ]:
pairs, _ = load_gt_pairs(config.TRAIN_GT)
val = pd.DataFrame([p for p in pairs if config.is_val_entity(p[0])], columns=["s1_id", "m_id"]).sample(100_000, random_state=0)
cols = ["entity_id", "business_name", "business_address", "country", "name_clean", "name_core", "addr_clean", "f_name_native"]
s1 = pd.read_parquet(config.CLEAN_DIR / "train_s1.parquet", columns=cols)
s23 = pd.concat([pd.read_parquet(config.CLEAN_DIR / f"train_{s}.parquet", columns=cols,
                                 filters=[("entity_id", "in", val.m_id.tolist())]) for s in ("s2", "s3")])
df = val.merge(s1, left_on="s1_id", right_on="entity_id").merge(s23, left_on="m_id", right_on="entity_id", suffixes=("", "_m"))
sim = lambda a, b: np.array([Indel.normalized_similarity(x, y) for x, y in zip(a, b)])
df["raw_name"] = sim(df.business_name.str.lower(), df.business_name_m.str.lower())
df["clean_name"] = sim(df.name_clean, df.name_clean_m)
df["core_name"] = sim(df.name_core, df.name_core_m)
df["raw_addr"] = sim(df.business_address.str.lower(), df.business_address_m.str.lower())
df["clean_addr"] = sim(df.addr_clean, df.addr_clean_m)
display(df.groupby(["country", "f_name_native_m"])[["raw_name", "clean_name", "core_name", "raw_addr", "clean_addr"]].mean().round(3))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, (r, c) in zip(ax, [("raw_name", "core_name"), ("raw_addr", "clean_addr")]):
    a.hist(df[r], bins=50, alpha=0.5, label=r); a.hist(df[c], bins=50, alpha=0.5, label=c); a.legend(); a.set_title("true-pair similarity")
plt.show()

In [ ]:
# Hardest remaining true pairs after cleaning -> next normalisation ideas
display(df.nsmallest(20, "core_name")[["business_name", "business_name_m", "name_core", "name_core_m", "core_name"]])